In [79]:
import numpy as np
import pandas as pd
import math

from matplotlib import pyplot as plt

from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn import metrics
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.model_selection import TimeSeriesSplit

In [80]:
def rmse(y_true, y_pred): 
    return math.sqrt(((y_true - y_pred)**2).mean())

In [81]:
def print_score(m, X_train, y_train, X_valid, y_valid):
    print('RMSE on train set: {:.4f}'.format(rmse(m.predict(X_train), y_train)))
    print('RMSE on valid set: {:.4f}'.format(rmse(m.predict(X_valid), y_valid)))
    print('R^2 on train set: {:.4f}'.format(m.score(X_train, y_train)))
    print('R^2 on valid set: {:.4f}'.format(m.score(X_valid, y_valid)))
    if hasattr(m, 'oob_score_'): print('R^2 on oob set: {:.4f}'.format(m.oob_score_))
    return

### What we need to make this work
Okay so we need to have a df converted to a feather_file with such 
- all the features( parameters ) that we can get
- the y value we want to predict  

In [82]:
df_treated = pd.read_feather('data/treated.feather')
df_treated

Price            Open                                      \
Ticker      crude_oil   US_dollar         gold     silver   
Date                                                        
2010-01-04  79.629997   77.930000  1117.699951  17.440001   
2010-01-05  81.629997   77.370003  1118.099976  17.781000   
2010-01-06  81.430000   77.650002  1135.900024  18.163000   
2010-01-07  83.199997   77.370003  1133.099976  18.333000   
2010-01-08  82.650002   77.980003  1138.199951  18.458000   
...               ...         ...          ...        ...   
2026-10-05  91.769997  101.919998  4169.399902  61.520000   
2026-10-06  89.269997  102.139999  4169.200195  60.919998   
2026-10-07  89.959999  101.889999  4195.000000  60.880001   
2026-10-08  89.000000  102.230003  4139.899902  59.063999   
2026-10-09  91.290001  102.129997  4159.700195  60.669998   

Price                                                                         \
Ticker     3month_treasury_yield    Nasdaq-100 10y_treasury_yield Volatility   
Date                                                                           
2010-01-04                 0.080   1882.689941              3.859  21.680000   
2010-01-05                 0.060   1887.280029              3.790  20.049999   
2010-01-06                 0.060   1887.910034              3.771  19.590000   
2010-01-07                 0.045   1878.160034              3.845  19.680000   
2010-01-08                 0.045   1870.900024              3.843  19.270000   
...                          ...           ...                ...        ...   
2026-10-05                 3.993  30812.759766              5.273  16.240000   
2026-10-06                 4.035  31255.400391              5.273  15.500000   
2026-10-07                 4.037  30976.250000              5.341  15.210000   
2026-10-08                 4.043  30985.230469              5.324  15.680000   
2026-10-09                 4.045  30900.060547              5.236  15.340000   

Price           Close              ...       High        Low              \
Ticker      crude_oil   US_dollar  ... Volatility  crude_oil   US_dollar   
Date                               ...                                     
2010-01-04  81.510002   77.529999  ...  21.680000  79.629997   77.260002   
2010-01-05  81.769997   77.620003  ...  20.129999  80.949997   77.089996   
2010-01-06  83.180000   77.489998  ...  19.680000  80.849998   77.360001   
2010-01-07  82.660004   77.910004  ...  19.709999  82.260002   77.300003   
2010-01-08  82.750000   77.470001  ...  19.270000  81.800003   77.349998   
...               ...         ...  ...        ...        ...         ...   
2026-10-05  89.430000  102.169998  ...  16.379999  88.739998  101.860001   
2026-10-06  89.440002  101.830002  ...  15.540000  86.860001  101.750000   
2026-10-07  88.279999  102.239998  ...  16.010000  87.959999  101.889999   
2026-10-08  91.489998  102.139999  ...  16.459999  88.769997  102.029999   
2026-10-09  91.849998  102.209999  ...  15.340000  90.010002  101.919998   

Price                                                                   \
Ticker             gold     silver 3month_treasury_yield    Nasdaq-100   
Date                                                                     
2010-01-04  1097.099976  16.870001                 0.055   1881.630005   
2010-01-05  1115.000000  17.575001                 0.040   1877.520020   
2010-01-06  1120.699951  17.840000                 0.045   1873.900024   
2010-01-07  1129.199951  18.090000                 0.040   1867.619995   
2010-01-08  1122.699951  18.120001                 0.035   1868.550049   
...                 ...        ...                   ...           ...   
2026-10-05  4150.399902  60.868999                 3.990  30798.419922   
2026-10-06  4130.700195  60.919998                 4.035  31208.199219   
2026-10-07  4091.199951  59.529999                 4.035  30904.460938   
2026-10-08  4128.100098  58.915001                 4.040  30556.400391 

In [83]:
df = df_treated.iloc[:,:32]
y = df_treated["up/down"]

# time series split 

In [84]:
tscv = TimeSeriesSplit(n_splits=3)

for train_index, test_index in tscv.split(df):
    X_train , X_test  = df.iloc[train_index] , df.iloc[test_index] 
    Y_train , Y_test   = y.iloc[train_index] , y.iloc[test_index]

In [85]:
'''
for train_index, val_index in tscv.split(df):
    X_train, X_val = df.iloc[train_index], df.iloc[val_index]
    Y_train, Y_val = y.iloc[train_index], y.iloc[val_index]'''

'\nfor train_index, val_index in tscv.split(df):\n    X_train, X_val = df.iloc[train_index], df.iloc[val_index]\n    Y_train, Y_val = y.iloc[train_index], y.iloc[val_index]'

In [88]:
model = RandomForestRegressor(n_estimators= 10, n_jobs=-1,oob_score=True)

%time model.fit(X_train, Y_train)

CPU times: user 550 ms, sys: 36.3 ms, total: 586 ms
Wall time: 220 ms


/opt/anaconda3/envs/NASDAQ_Prediction-Project/lib/python3.14/site-packages/sklearn/ensemble/_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.r2_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requir

In [89]:
print_score(model, X_train , Y_train, X_test,Y_test)

RMSE on train set: 0.1903
RMSE on valid set: 0.5144
R^2 on train set: 0.8538
R^2 on valid set: -0.0634
R^2 on oob set: -0.0203
